<div style="border:2px solid red; padding:10px; background-color:#ffe6e6;">
<b>TODO:</b> 
    - Correct for host id  
    - Fill in wget for result retrieval  
    - Add which analysis parameters we used + explanation
    - References
    - Answer research question at the end
    - adonis permdisp for beta diversity
</div>

| Plugin                       | What it tests / produces   | Intuition                         | When to use                |
| ---------------------------- | -------------------------- | --------------------------------- | -------------------------- |
| **beta / beta-phylogenetic** | Distance matrix            | How far apart samples are         | Always (prerequisite)      |
| **pcoa**                     | Ordination (PCoA)          | Map of sample relationships       | For visualization          |
| **beta-group-significance**  | PERMANOVA, PERMDISP        | Do groups differ?                 | For hypothesis testing     |
| **mantel**                   | Matrix correlation         | Do two distances correlate?       | Environmental associations |
| **procrustes-analysis**      | Compare ordinations        | Do two maps agree?                | Compare metrics/datasets   |
| **procrustes-plot**          | Visualization of alignment | Show how maps overlap             | Publishable figure         |
| **beta-rarefaction**         | Rarefaction curves         | Does depth affect beta diversity? | QC / robustness            |

Know differences: no bray-curtis, jaccard


| Metric                 | Uses abundance? | Uses phylogeny? | Best for                                         |
| ---------------------- | --------------- | --------------- | ------------------------------------------------ |
| **Jaccard**            | ❌ No            | ❌ No            | Presence/absence questions; rare taxa            |
| **Bray–Curtis**        | ✔ Yes           | ❌ No            | Gut studies, abundance differences               |
| **Unweighted UniFrac** | ❌ No            | ✔ Yes           | Big ecological shifts; turnover                  |
| **Weighted UniFrac**   | ✔ Yes           | ✔ Yes           | Subtle, abundance-based phylogenetic differences |

Are rare taxa driving differences? (Jaccard, Unweighted UniFrac)

Are abundant taxa driving differences? (Bray, Weighted UniFrac)

Are phylogenetic differences important? (UniFrac metrics)

# 5. Diversity analysis


This notebook performs the diversity analysis. We start by generating alpha rarefaction curves to evaluate sequencing depth and determine an appropriate rarefaction level, followed by beta rarefaction to assess the stability of beta diversity patterns under repeated subsampling. Based on these results, we compute the full set of alpha and beta diversity metrics. The pipeline automatically organizes all outputs into a consistent folder structure and loads the key visualizations for inspection. It also allows fast exploration of different trends in the cohort, since you can easily subset the data (e.g. by age, country, or milk type) by adjusting the filter settings and rerunning only the diversity analysis steps.

### Notebook Structure

**1.** Alpha rarefaction  
**2.** Beta rarefaction  
**3.** Diversity analysis  
&nbsp;&nbsp;&nbsp;&nbsp;**3.1** Setting up analysis parameters  
&nbsp;&nbsp;&nbsp;&nbsp;**3.2** Running alpha and beta analysis  
&nbsp;&nbsp;&nbsp;&nbsp;**3.3** Organizing outputs  
**4.** Visualizing results of the diversity analysis


<div style="border: 2px solid #1f77b4; padding: 10px; border-radius: 6px; background-color: #eaf3fb;">
    
<b> Research questions this notebook answers:</b>
- How does alpha diversity (e.g., richness, Shannon diversity) vary with age and across metadata groups such as feeding pattern, delivery mode, treatment, and geographic location?
- How does beta diversity (community composition differences) vary with age and between metadata groups, and which factors explain the largest share of variation?

</div>


### Import Packages

In [1]:
# Import all necessary packages
import os
import IPython
import pandas as pd
import matplotlib.pyplot as plt
import qiime2 as q2
from qiime2 import Visualization
import shutil
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import Dropdown, FloatSlider, Checkbox, VBox, HBox, IntSlider

%matplotlib inline

### Set Working Directory
Ensure that the working directory is correctly set to the 'scripts' folder within the main project directory. 
Otherwise, the file paths used in this notebook may not work properly.

In [2]:
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts").


In [3]:
# 3 - Data directories
# metadata_dir = "../data/raw"
meta_data_dir = "../data/processed/metadata"
raw_data_dir = "../data/raw"
denoising_data_dir = "../data/processed/denoising"
taxonomy_data_dir = "../data/processed/taxonomy"
phylogeny_data_dir = "../data/processed/phylogeny"
diversity_data_dir = "../data/processed/diversity"

In [4]:
%%bash -s "$diversity_data_dir"
mkdir -p "$1"

### Set run and metric to work with

In [5]:
# Rarefaction depth options
depth_selector = IntSlider(
    value=14000,          # default
    min=5000,
    max=20000,
    step=1000,            # change in increments of 1000
    description="Depth:",
    continuous_update=False
)

# Switch sample filtering on/off
filter_toggle = Checkbox(
    value=False,
    description="Enable filtering"
)

# Age filter: only three possible values
age_selector = Dropdown(
    options=[None, 4.0, 7.0, 10.0],
    value=None,
    description="Age (months):"
)

# Country filter
country_selector = Dropdown(
    options=[None, "Russia", "Finland"],
    value=None,
    description="Country:"
)

# Milk type filter
milk_selector = Dropdown(
    options=[None, "bd", "fd"],
    value=None,
    description="Milk type:"
)

VBox([
    depth_selector,
    filter_toggle,
    age_selector,
    country_selector,
    milk_selector
])

In [6]:
# Read values from the filtering + depth widgets
sampling_depth = depth_selector.value
enable_sample_filtering = filter_toggle.value
filter_age_months = age_selector.value
filter_country = country_selector.value
filter_milk_type = milk_selector.value

print("Running core microbiota pipeline with:")
print(sampling_depth, enable_sample_filtering, filter_age_months, filter_country, filter_milk_type)

Running core microbiota pipeline with:
14000 False None None None


## 1. Alpha Rarefaction 
First, we perform alpha rarefaction to evaluate how sequencing depth influences within-sample diversity and to choose a suitable rarefaction depth for downstream analyses. To select the minimum and maximum depths, we inspect the sequencing depth distribution in the feature table and base our rarefaction range on those values.


In [7]:
! qiime feature-table summarize \
    --i-table $denoising_data_dir/dada2_table.qza \
    --m-sample-metadata-file $meta_data_dir/metadata_merged.tsv \
    --o-visualization $denoising_data_dir/dada2_table.qzv

QIIME is caching your current deployment for improved performance. This may take a few moments and should only happen once per deployment.
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/denoising/dada2_table.qzv


In [8]:
Visualization.load(f"{denoising_data_dir}/dada2_table.qzv")

<visualization: Visualization uuid: 9ab06b1d-e975-40c0-946c-620d02a9d444>

We kept the minimum sampling depth at 1 as a reference point, even though values below ~5,000 reads are not biologically meaningful for this dataset. Based on the sequencing depth distribution in the feature table, we selected a maximum depth of 55,000 reads to retain at least three-quarters of the samples. A step size of 15 was used to provide a smooth progression of rarefaction levels across this range.

In [9]:
! qiime diversity alpha-rarefaction \
    --i-table $denoising_data_dir/dada2_table.qza \
    --p-max-depth 55000 \
    --p-steps 15 \
    --m-metadata-file $meta_data_dir/metadata_merged.tsv \
    --o-visualization $diversity_data_dir/alpha-rarefaction.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/diversity/alpha-rarefaction.qzv


In [10]:
Visualization.load(f"{diversity_data_dir}/alpha-rarefaction.qzv")

<visualization: Visualization uuid: ab6fce46-21c9-4d14-a15a-3709e60015b4>

## 2. Beta Rarefaction 

Next, we perform beta rarefaction to evaluate how stable between-sample diversity patterns are under repeated subsampling. By rarefying the data multiple times and recalculating Bray–Curtis distances, we can assess whether the overall clustering structure is robust or sensitive to sequencing-depth variation. For this analysis, we used a sampling depth of 14,000 reads, which was chosen based on the sequencing-depth distribution and the alpha-rarefaction curves. This depth retains most samples while providing sufficient coverage, ensuring that the observed beta-diversity patterns reflect true biological structure rather than artifacts of uneven sequencing depth.

In [11]:
! qiime diversity beta-rarefaction \
  --i-table $denoising_data_dir/dada2_table.qza \
  --p-metric braycurtis \
  --p-clustering-method upgma \
  --m-metadata-file $meta_data_dir/metadata_merged.tsv \
  --p-sampling-depth {sampling_depth} \
  --p-iterations 10 \
  --o-visualization $diversity_data_dir/beta-rarefaction-braycurtis.qzv \
  --verbose

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/conda/lib/python3.10/site-packages/skbio/stats/ordination/_principal_coordinate_analysis.py:146: RuntimeWarning: The result contains negative eigenvalues. Please compare their magnitude with the magnitude of some of the largest positive eigenvalues. If the negative ones are smaller, it's probably safe to ignore them, but if they are large in magnitude, the results won't be useful. See the Notes section for more details. The smallest eigenvalue is -0.41420684929593954 and the largest is 12.519747675771914.
  warn(
/opt/conda/lib/python3.10/site-packages/skbio/stats/ordination/_principal_coordinate_analysis.py:146: RuntimeWarning: The result contains negative e

In [12]:
Visualization.load(f"{diversity_data_dir}/beta-rarefaction-braycurtis.qzv")

<visualization: Visualization uuid: 700a6505-9a7b-4f33-a508-d9f3079c51d9>

The PCoA shows a stable clustering pattern across iterations, indicating that rarefying to 14,000 reads does not distort the beta-diversity structure.

## 3. Diversity Analysis

### 3.1 Setting Up the Analysis Parameters

These are the settings used for this diversity analysis. Here we define the rarefaction depth and optional filtering choices. Changing these values will affect the downstream core-metrics results.

In [13]:
# Input files
table_raw = f"{denoising_data_dir}/dada2_table.qza"
tree_qza  = f"{phylogeny_data_dir}/sepp-tree.qza"
metadata  = f"{meta_data_dir}/metadata_merged.tsv"

Based on the filtering settings above, we create a short label and a folder where all outputs from this run will be stored. This helps to keep different runs organized.

In [14]:
# Build a short label describing the filtering settings

filter_label_parts = []

if enable_sample_filtering:
    if filter_age_months:
        filter_label_parts.append(f"age-{filter_age_months}")
    if filter_country:
        filter_label_parts.append(f"country-{filter_country}")
    if filter_milk_type:
        filter_label_parts.append(f"milk-{filter_milk_type}")

# If nothing was added, mark it as "no-filter"
filter_label = "_".join(filter_label_parts) if filter_label_parts else "no-filter"

# Directory for storing all results of this run
run_dir = f"{diversity_data_dir}/depth-{sampling_depth}_{filter_label}"

!mkdir {run_dir}
print("Run directory:", run_dir)

mkdir: cannot create directory ‘../data/processed/diversity/depth-14000_no-filter’: File exists
Run directory: ../data/processed/diversity/depth-14000_no-filter


If filtering is turned on, samples are selected using metadata conditions (age, country, or milk type). Otherwise, we use the full table. The filtered or unfiltered table is then passed to the main diversity pipeline.

In [15]:
# Apply sample filtering based on metadata (if activated)

if enable_sample_filtering:
    print("Filtering enabled...")

    filtered_table = f"{run_dir}/filtered_table.qza"
    where_conditions = []

    # Add conditions depending on what the user set
    if filter_age_months is not None:
        where_conditions.append(f"age_months = {filter_age_months}")
    if filter_country is not None:
        where_conditions.append(f"geo_location_name = '{filter_country}'")
    if filter_milk_type is not None:
        where_conditions.append(f"diet_milk = '{filter_milk_type}'")

    # Combine everything into a single WHERE statement
    where_sql = " AND ".join(where_conditions)
    print("WHERE:", where_sql)

    # Run QIIME2 filtering
    !qiime feature-table filter-samples \
        --i-table {table_raw} \
        --m-metadata-file {metadata} \
        --p-where "{where_sql}" \
        --o-filtered-table {filtered_table}

    table_for_run = filtered_table

else:
    print("No filtering applied.")
    table_for_run = table_raw

No filtering applied.


### 3.2 Running Alpha and Beta Analysis

In this step we run QIIME2’s core-metrics-phylogenetic pipeline.
This calculates all standard diversity metrics (alpha and beta) at the chosen rarefaction depth.
The results are stored inside the run directory so we can generate visualizations later.

In [16]:
print("Running core metrics...")

!qiime diversity core-metrics-phylogenetic \
  --i-table {table_for_run} \
  --i-phylogeny {tree_qza} \
  --m-metadata-file {metadata} \
  --p-sampling-depth {sampling_depth} \
  --output-dir {run_dir}/core_metrics

Running core metrics...
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureTable[Frequency] to: ../data/processed/diversity/depth-14000_no-filter/core_metrics/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: ../data/processed/diversity/depth-14000_no-filter/core_metrics/faith_pd_vector.qza
Saved SampleData[AlphaDiversity] to: ../data/processed/diversity/depth-14000_no-filter/core_metrics/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: ../data/processed/diversity/depth-14000_no-filter/core_metrics/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: ../data/processed/diversity/depth-14000_no-filter/core_metrics/evenness_vector.qza
Saved DistanceMatrix to: ../data/processed/d

After the core metrics are computed, we generate the corresponding alpha and beta diversity visualizations.
These include group significance tests for alpha metrics and emperor plots for the beta diversity PCoA results.

In [17]:
print("Generating alpha & beta visualizations...")

# Alpha diversity visualizations
alpha_vis_commands = [
    ("faith_pd_vector.qza",           "faith-pd-group-significance.qzv"),
    ("shannon_vector.qza",            "shannon-group-significance.qzv"),
    ("observed_features_vector.qza",  "observed-features-group-significance.qzv"),
    ("evenness_vector.qza",           "evenness-group-significance.qzv")
]

for infile, outfile in alpha_vis_commands:
    input_path = f"{run_dir}/core_metrics/{infile}"
    output_path = f"{run_dir}/core_metrics/{outfile}"
    
    if os.path.exists(input_path):
        !qiime diversity alpha-group-significance \
            --i-alpha-diversity {input_path} \
            --m-metadata-file {metadata} \
            --o-visualization {output_path}

print("Finished alpha visualizations!")


# Beta diversity emperor plots
beta_vis_commands = [
    ("unweighted_unifrac_pcoa_results.qza", "unweighted_unifrac_emperor.qzv"),
    ("weighted_unifrac_pcoa_results.qza",   "weighted_unifrac_emperor.qzv"),
    ("jaccard_pcoa_results.qza",            "jaccard_emperor.qzv"),
    ("bray_curtis_pcoa_results.qza",        "bray_curtis_emperor.qzv")
]

for pcoa_file, vis_file in beta_vis_commands:
    pcoa_path = f"{run_dir}/core_metrics/{pcoa_file}"
    output_path = f"{run_dir}/core_metrics/{vis_file}"
    
    if os.path.exists(pcoa_path):
        !qiime emperor plot \
            --i-pcoa {pcoa_path} \
            --m-metadata-file {metadata} \
            --o-visualization {output_path}

print("Finished beta visualizations!")

Generating alpha & beta visualizations...
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/diversity/depth-14000_no-filter/core_metrics/faith-pd-group-significance.qzv
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/diversity/depth-14000_no-filter/core_metrics/shannon-group-significance.qzv
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: Us

### 3.3 Organizing Alpha and Beta Diversity Outputs

After generating the core metrics and visualizations, we sort the outputs into separate folders for alpha and beta diversity.
This keeps the run directory clean and makes it easier to find the files later.

In [23]:
# Create folders for storing metrics and visualizations

alpha_metrics_dir = f"{run_dir}/alpha/metrics"
alpha_vis_dir     = f"{run_dir}/alpha/visualizations"
beta_metrics_dir  = f"{run_dir}/beta/metrics"
beta_vis_dir      = f"{run_dir}/beta/visualizations"

# Make the directories if they don't already exist
for d in [alpha_metrics_dir, alpha_vis_dir, beta_metrics_dir, beta_vis_dir]:
    os.makedirs(d, exist_ok=True)

Now we move the alpha diversity results (both the numeric vectors and the QZV visualizations) into the corresponding folders.

In [19]:
# Move alpha diversity artifacts (metrics)
alpha_files = [
    "faith_pd_vector.qza",
    "shannon_vector.qza",
    "observed_features_vector.qza",
    "evenness_vector.qza"
]

for fname in alpha_files:
    src = f"{run_dir}/core_metrics/{fname}"
    if os.path.exists(src):
        shutil.move(src, f"{alpha_metrics_dir}/{fname}")

# Move alpha diversity visualizations
for _, vis_name in alpha_vis_commands:
    src = f"{run_dir}/core_metrics/{vis_name}"
    if os.path.exists(src):
        shutil.move(src, f"{alpha_vis_dir}/{vis_name}")

We do the same for beta diversity: distance matrices, PCoA results, and their emperor plots.

In [20]:
# Move beta diversity artifacts (distance matrices + PCoA files)
beta_files = [
    "unweighted_unifrac_distance_matrix.qza",
    "unweighted_unifrac_pcoa_results.qza",
    "weighted_unifrac_distance_matrix.qza",
    "weighted_unifrac_pcoa_results.qza",
    "bray_curtis_distance_matrix.qza",
    "bray_curtis_pcoa_results.qza",
    "jaccard_distance_matrix.qza",
    "jaccard_pcoa_results.qza"
]

for fname in beta_files:
    src = f"{run_dir}/core_metrics/{fname}"
    if os.path.exists(src):
        shutil.move(src, f"{beta_metrics_dir}/{fname}")

# Move beta emperor visualizations
for _, vis_name in beta_vis_commands:
    src = f"{run_dir}/core_metrics/{vis_name}"
    if os.path.exists(src):
        shutil.move(src, f"{beta_vis_dir}/{vis_name}")

The rarefied table is still useful, so we keep it in the main run directory.
After that, we delete the leftover core_metrics/ folder to reduce clutter.

In [21]:
print(f"../{run_dir}/alpha-rarefaction.qzv")

../../data/processed/diversity/depth-14000_no-filter/alpha-rarefaction.qzv


In [22]:
# Move rarefied table to the run directory (keep this file)
rarefied_table = f"{run_dir}/core_metrics/rarefied_table.qza"
if os.path.exists(rarefied_table):
    shutil.move(rarefied_table, run_dir)

alpha_rarefaction = f"{diversity_data_dir}/alpha-rarefaction.qzv"
if os.path.exists(alpha_rarefaction):
    shutil.move(alpha_rarefaction, run_dir)   
    
# Remove the now-empty core_metrics folder
!rm -r {run_dir}/core_metrics

Error: Destination path '../data/processed/diversity/depth-14000_no-filter/rarefied_table.qza' already exists

### Optional Cleanup: Remove an Old Run Directory

This block is optional. If you want to re-run the pipeline from scratch with the same parameters, you can delete the existing run directory first. The code is commented out to avoid accidentally removing data.

In [ ]:
# ======================================================
# DELETE RUN DIRECTORY 
# ======================================================
# Uncomment the lines below to delete the entire run_dir.
# This is useful if you want to re-run the pipeline cleanly
# with the same sampling depth or filters.
# ======================================================

# #folder_to_delete = run_dir
# folder_to_delete = f"{diversity_data_dir}/depth-14000_no-filter"

# if os.path.exists(folder_to_delete):
#     print("Fast deletion with verbose output...\n")
#     !rm -rfv {folder_to_delete}
#     print("Directory deleted.")
# else:
#     print("Unsafe path — aborting.")

## 4. Visualize Results of Diversity Analysis

Here we load the .qzv files that were generated in the previous steps.
You can switch the folder paths if you want to inspect results from a different run.
Only a few visualizations are shown by default; others can be uncommented if needed

In [25]:
# Paths to the visualization folders
# (you can replace these with another run if you want)

alpha_to_visualize = alpha_vis_dir
# alpha_to_visualize = f"{diversity_data_dir}/depth-10000_no-filter/alpha/visualizations"

beta_to_visualize = beta_vis_dir
# beta_to_visualize =  f"{diversity_data_dir}/depth-10000_no-filter/beta/visualizations"

In [26]:
# --- Alpha diversity visualizations ---

# Visualization.load(f"{alpha_to_visualize}/faith-pd-group-significance.qzv")
# Visualization.load(f"{alpha_to_visualize}/shannon-group-significance.qzv")
# Visualization.load(f"{alpha_to_visualize}/observed-features-group-significance.qzv")
Visualization.load(f"{alpha_to_visualize}/evenness-group-significance.qzv")

<visualization: Visualization uuid: cf469104-0050-4156-bb87-fd7a040779b8>

In [29]:
# --- Beta diversity emperor plots ---

#Visualization.load(f"{beta_to_visualize}/unweighted_unifrac_emperor.qzv")
Visualization.load(f"{beta_to_visualize}/weighted_unifrac_emperor.qzv")
# Visualization.load(f"{beta_to_visualize}/jaccard_emperor.qzv")
# Visualization.load(f"{beta_to_visualize}/bray_curtis_emperor.qzv")

<visualization: Visualization uuid: 30b6d757-bb2f-4c60-801b-0c2c61766340>

<div style="border: 2px solid #1f77b4; padding: 10px; border-radius: 6px; background-color: #eaf3fb;">
<b> Answers to the research questions:</b>

- How does alpha diversity (e.g., richness, Shannon diversity) vary with age and across metadata groups such as feeding pattern, delivery mode, treatment, and geographic location?
  - <!-- Add bullet-point answers here -->

- How does beta diversity (community composition differences) vary with age and between metadata groups, and which factors explain the largest share of variation?
  - <!-- Add bullet-point answers here -->

</div>


# References

QIIME 2 Documentation. (2024). core-metrics-phylogenetic: Core diversity metrics (phylogenetic and non-phylogenetic). https://docs.qiime2.org/2024.10/plugins/available/diversity/core-metrics-phylogenetic/

National Cancer Institute, Center for Cancer Research. QIIME 2: Lesson 5 — Microbial Diversity and Core Metrics. https://bioinformatics.ccr.cancer.gov/docs/qiime2/Lesson5/
